# 선박 매칭 P9 — Colab GPU 학습 노트북 (T9.3 언어모델)

위에서부터 순서대로 실행하면 된다. 전체 약 1시간 (L4 기준).

| 단계 | 하는 일 | 어디서 | 시간 (L4) |
|---|---|---|---|
| 0 | GPU 확인, 번들 풀기, 패키지 설치 | | 10분 |
| 1 | fold별 LoRA 어댑터 5개 학습 → 한글 후보 채점 (표본 밖 점수) | GPU | 15분 |
| 2 | **검증**: 언어모델 피처를 넣은 5-fold CV. 기준 0.9851보다 높으면 채택 | CPU | 20분 |
| 3 | 최종 언어모델 (382쌍 전부로 학습, 병합 저장) | GPU | 3분 |
| 4 | 최종 재순위 모델 (train 전체) | CPU | 5분 |
| 5 | 제출 리허설 (서버와 같은 버전, 3,000행) | GPU | 5분 |
| 6 | `submit_llm.zip`(약 4GB) → Google Drive 저장 | | 5분 |

**준비물**: `colab_bundle_p9.zip`을 Google Drive `내 드라이브`에 올려 둔다.
**런타임**: `런타임 → 런타임 유형 변경 → L4 GPU`(또는 A100). T4도 돌아가지만 bf16이 느려 2~3배 오래 걸린다.
런타임이 끊기면 0단계부터 다시 실행한다. 1단계 결과는 Drive에 저장되므로 `SKIP_DONE=True`면 다시 계산하지 않는다.

## 0. 준비

In [1]:
!nvidia-smi -L
import os, sys, zipfile, subprocess, time, shutil, glob
# Where the bundle comes from:
#   "drive"  : colab_bundle_p9.zip in My Drive (results are also saved to Drive and survive restarts)
#   "upload" : pick the file in the upload dialog (Colab in the browser)
#   "local"  : the file is already on the runtime, e.g. dragged into the file panel -> /content/colab_bundle_p9.zip
SOURCE = "drive"
if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    BUNDLE, SAVE = "/content/drive/MyDrive/colab_bundle_p9.zip", "/content/drive/MyDrive/p9_results"
elif SOURCE == "upload":
    from google.colab import files
    BUNDLE, SAVE = "/content/" + next(iter(files.upload())), "/content/p9_results"
else:
    BUNDLE, SAVE = "/content/colab_bundle_p9.zip", "/content/p9_results"
os.makedirs(SAVE, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall("/content/open_data")
os.chdir("/content/open_data")
!ls

Mounted at /content/drive
open  plans  src  submit


### 0-1. 패키지 두 벌 설치
- **Colab 기본 파이썬**: 언어모델 학습용 (`transformers`, `peft`). Colab에 깔린 GPU용 torch를 그대로 쓴다.
- **`/content/srv` 가상환경**: 평가 서버와 같은 버전 (Python 3.11, numpy 1.26.4, scikit-learn 1.5.2, torch 2.7.1 …).
  재순위 모델(`ranker.pkl`)은 반드시 여기서 학습해야 서버에서 열린다. numpy 2.x에서 저장한 pickle은 서버에서 로드 실패한다 (T9.1에서 확인).

In [6]:
!pip -q install transformers==4.57.6 peft==0.21.2 accelerate==1.9.0 sentencepiece uv
!pip -q uninstall -y torchao  # Colab's torchao 0.10 breaks peft's LoRA dispatch; unused here
!uv venv -q --clear -p 3.11 /content/srv
!uv pip install -q -p /content/srv/bin/python numpy==1.26.4 pandas==2.0.3 scikit-learn==1.5.2 joblib==1.5.2 scipy==1.15.3 \
    torch==2.7.1 transformers==4.57.6 accelerate==1.9.0 safetensors==0.6.2 sentencepiece==0.2.0
!/content/srv/bin/python -c "import sys, numpy, sklearn, torch; print(sys.version.split()[0], numpy.__version__, sklearn.__version__, torch.__version__, torch.cuda.is_available())"
SRV = "/content/srv/bin/python"

3.11.17 1.26.4 1.5.2 2.7.1+cu126 True


In [3]:
# Base model (Apache-2.0, general-purpose pretrained weights are allowed by the rules), ~4.4GB
from huggingface_hub import snapshot_download
os.environ["P8_WORKDIR"] = "/content/llmwork"
os.environ["KANANA_DIR"] = snapshot_download("kakaocorp/kanana-1.5-2.1b-instruct-2505", local_dir="/content/llmwork/kanana")
LLM = "plans/scratch/p9/llm"
SCORES = f"{LLM}/scores"
os.makedirs(SCORES, exist_ok=True)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

kanana-logo.png:   0%|          | 0.00/59.0k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.63G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

## 1. fold별 LoRA 어댑터 학습 + 채점
한글 이름 382개를 선박 단위 5-fold로 나눠, fold k의 이름은 **나머지 4개 fold로만 학습한 어댑터**로 채점한다.
이렇게 해야 2단계 CV와 최종 모델 학습에 쓰는 점수가 "처음 보는 이름"에 대한 점수가 된다 (정답 유출 방지).
출력의 `pairs_per_sec`가 채점 속도다 (참고: Mac MPS 36쌍/초).

In [7]:
SKIP_DONE = True
for k in range(5):
    out = f"{SCORES}/f{k}_fwd_s1.jsonl"
    saved = f"{SAVE}/f{k}_fwd_s1.jsonl"
    if SKIP_DONE and os.path.exists(saved):
        shutil.copy(saved, out); print("fold", k, "restored from Drive"); continue
    t0 = time.time()
    r = subprocess.run([sys.executable, "-u", "-W", "ignore", "lora_run.py", "--fold", str(k), "--direction", "fwd", "--seed", "1"],
                       cwd=LLM, capture_output=True, text=True)
    print("fold", k, "exit", r.returncode, f"{time.time() - t0:.0f}s", (r.stdout.strip().splitlines() or [r.stderr[-2000:]])[-1])
    if r.returncode == 0:
        shutil.copy(out, saved)

fold 0 exit 0 51s {"tag": "f0_fwd_s1", "n_train": 302, "train_sec": 31.96744179725647, "score_pairs": 4983, "score_sec": 4.129514455795288, "pairs_per_sec": 1206.6793937497773}
fold 1 exit 0 52s {"tag": "f1_fwd_s1", "n_train": 305, "train_sec": 33.017672300338745, "score_pairs": 4809, "score_sec": 4.052671432495117, "pairs_per_sec": 1186.6246943782542}
fold 2 exit 0 51s {"tag": "f2_fwd_s1", "n_train": 309, "train_sec": 32.549304485321045, "score_pairs": 4556, "score_sec": 3.8496201038360596, "pairs_per_sec": 1183.4934037932858}
fold 3 exit 0 51s {"tag": "f3_fwd_s1", "n_train": 304, "train_sec": 31.84447717666626, "score_pairs": 4919, "score_sec": 4.082422733306885, "pairs_per_sec": 1204.9217637036481}
fold 4 exit 0 52s {"tag": "f4_fwd_s1", "n_train": 308, "train_sec": 32.83729553222656, "score_pairs": 4637, "score_sec": 3.914026975631714, "pairs_per_sec": 1184.7133473707345}


In [8]:
# Name-only check per fold: does the LLM alone pick the right English name? (fold 0 on Mac: 80/80)
import json, numpy as np
for f in sorted(glob.glob(f"{SCORES}/f*_fwd_s1.jsonl")):
    rows = [json.loads(l) for l in open(f)]
    ok = sum(int(np.argmax(r["scores"]) == r["cands"].index(r["target_idx"])) for r in rows)
    print(os.path.basename(f), f"{ok}/{len(rows)}")

f0_fwd_s1.jsonl 80/80
f1_fwd_s1.jsonl 77/77
f2_fwd_s1.jsonl 73/73
f3_fwd_s1.jsonl 77/78
f4_fwd_s1.jsonl 74/74


## 2. 검증 — 언어모델 피처 5-fold CV
`mean`의 `weighted`를 언어모델 없는 기준(**0.9851**, hangul 0.9794 / non 0.9984)과 비교한다.
- 높으면 → 3~6단계로 진행해 `submit_llm.zip`을 만든다.
- 같거나 낮으면 → 채택하지 않는다. 로컬에 이미 있는 `submit.zip`(언어모델 없음)을 제출한다.

결과 숫자는 Claude에게 알려 주면 plans/P9.md에 기록한다.

In [ ]:
!{SRV} src/train.py --cv --llm {SCORES} | tee {SAVE}/cv_llm.out

## 3. 최종 언어모델 — 382쌍 전부로 학습, LoRA 병합 후 `submit/model/llm/`에 저장 (약 4.2GB)

In [9]:
r = subprocess.run([sys.executable, "-u", "-W", "ignore", "lora_run.py", "--fold", "-1", "--direction", "fwd", "--seed", "1",
                    "--save_merged", "/content/open_data/submit/model/llm"], cwd=LLM, capture_output=True, text=True)
print(r.returncode, r.stdout.strip().splitlines()[-1:] or r.stderr[-2000:])
!du -sh submit/model/llm && ls submit/model/llm

0 ['{"tag": "f-1_fwd_s1", "n_train": 382, "train_sec": 39.85147190093994, "saved": "/content/open_data/submit/model/llm"}']
4.0G	submit/model/llm
chat_template.jinja	model.safetensors	 tokenizer.json
config.json		special_tokens_map.json
generation_config.json	tokenizer_config.json


## 4. 최종 재순위 모델 — train 전체, 서버 버전 환경에서 학습
train 행의 언어모델 피처는 1단계의 표본 밖 점수를 그대로 쓴다.

In [10]:
!{SRV} -W error::UserWarning src/train.py --full submit/model/ranker.pkl --llm {SCORES}
!cp src/matcher.py src/normalize.py src/llm_scorer.py submit/model/ && ls -la submit/model

saved submit/model/ranker.pkl (use_llm=True) in 358s
total 656
drwxr-xr-x 3 root root   4096 Oct 14 05:33 .
drwxr-xr-x 3 root root   4096 Oct 14 05:05 ..
drwxr-xr-x 2 root root   4096 Oct 14 05:27 llm
-rw-r--r-- 1 root root   3248 Oct 14 05:33 llm_scorer.py
-rw-r--r-- 1 root root  14160 Oct 14 05:33 matcher.py
-rw-r--r-- 1 root root   7599 Oct 14 05:33 normalize.py
-rw-r--r-- 1 root root 629962 Oct 14 05:33 ranker.pkl


## 5. 제출 리허설
서버와 같은 폴더 구조로 `script.py`를 서버 버전 환경(GPU)에서 실행한다. 가짜 test = train 3,000행 (call_sign을 처음 보는 값으로 바꿔 모델 경로를 탄다).
학습에 쓴 행이라 점수는 참고용이고, 확인할 것은 **exit 0, 형식 통과, 시간** (서버 L4 제한 20분 → 5분 이하면 안전).

In [11]:
!cd submit && rm -rf model/__pycache__ && zip -q -r -1 -X /content/submit_llm.zip script.py requirements.txt model && cd .. && unzip -l /content/submit_llm.zip
shutil.rmtree("/content/rehearsal", ignore_errors=True)
!python plans/scratch/p9/rehearse.py /content/submit_llm.zip /content/rehearsal {SRV} 3000

Archive:  /content/submit_llm.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
     1663  2026-10-14 05:05   script.py
       83  2026-10-14 05:05   requirements.txt
        0  2026-10-14 05:33   model/
   629962  2026-10-14 05:33   model/ranker.pkl
     3248  2026-10-14 05:33   model/llm_scorer.py
    14160  2026-10-14 05:33   model/matcher.py
        0  2026-10-14 05:27   model/llm/
    51158  2026-10-14 05:27   model/llm/tokenizer_config.json
 17210524  2026-10-14 05:27   model/llm/tokenizer.json
      147  2026-10-14 05:27   model/llm/generation_config.json
    14287  2026-10-14 05:27   model/llm/chat_template.jinja
      709  2026-10-14 05:27   model/llm/config.json
4174003008  2026-10-14 05:27   model/llm/model.safetensors
      444  2026-10-14 05:27   model/llm/special_tokens_map.json
     7599  2026-10-14 05:33   model/normalize.py
---------                     -------
4191936992                     15 files
Saved 3000 predictions to output/submission.c

## 6. Drive에 저장 → 내려받아 데이콘에 제출
`내 드라이브/p9_results/submit_llm.zip`이 생긴다. 4GB라 브라우저 직접 다운로드보다 Drive에서 받는 편이 안정적이다.
데이콘 제출 한도(하루 3회)를 고려해, 리허설이 통과한 zip만 올린다.

In [12]:
shutil.copy("/content/submit_llm.zip", f"{SAVE}/submit_llm.zip")
!ls -la {SAVE}

total 3262193
drwx------  2 root root       4096 Oct 14 05:43 .
drwx------ 43 root root       4096 Oct 14 05:05 ..
-rw-------  1 root root     141728 Oct 14 05:19 f0_fwd_s1.jsonl
-rw-------  1 root root     136461 Oct 14 05:20 f1_fwd_s1.jsonl
-rw-------  1 root root     129440 Oct 14 05:21 f2_fwd_s1.jsonl
-rw-------  1 root root     139481 Oct 14 05:22 f3_fwd_s1.jsonl
-rw-------  1 root root     131832 Oct 14 05:23 f4_fwd_s1.jsonl
-rw-------  1 root root 3339797266 Oct 14 05:43 submit_llm.zip
